In [1]:
import os, pandas as pd
os.chdir(os.path.expanduser("~/growth-marketing-analytics"))
ga = pd.read_parquet("data/processed/ga_sessions.parquet")

def stats(df):
    return [len(df), int((df.transactions > 0).sum()), round(df.revenue.sum(), 2)]

rows = [
    ["Paid Search (total)"] + stats(ga[ga.channel == "Paid Search"]),
    ["AW - Dynamic Search Ads"] + stats(ga[ga.campaign == "AW - Dynamic Search Ads Whole Site"]),
    ["AW - Accessories"] + stats(ga[ga.campaign == "AW - Accessories"]),
]
pd.DataFrame(rows, columns=["segment","sessions","buying_sessions","revenue"])

,segment,sessions,buying_sessions,revenue
0,Paid Search (total),25326,469,43558.90
1,AW - Dynamic Search Ads,6213,138,10787.36
2,AW - Accessories,5327,98,13997.55


In [2]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

wb = Workbook(); ws = wb.active; ws.title = "Scenario"
bold = Font(bold=True); inp = PatternFill("solid", fgColor="FFF2CC")

ws["A1"] = "Paid Search ROAS / CAC Scenario Analysis"; ws["A1"].font = Font(bold=True, size=14)
ws["A2"] = "Advertising spend is ASSUMED for scenario analysis and is not part of the original dataset. Sessions are used as a proxy for paid clicks."
ws["A4"] = "Gross margin (assumption)"; ws["B4"] = 0.5; ws["B4"].fill = inp; ws["B4"].number_format = "0%"

headers = ["Segment","Sessions (≈clicks)","Buying sessions","Revenue $","Assumed CPC $",
           "Spend $","ROAS","CAC $","ROMI","Break-even CPC (revenue)","Break-even CPC (profit)"]
for c, h in enumerate(headers, 1):
    ws.cell(row=6, column=c, value=h).font = bold

for i, (seg, s, b, r) in enumerate(rows, start=7):
    ws[f"A{i}"], ws[f"B{i}"], ws[f"C{i}"], ws[f"D{i}"] = seg, s, b, r
    ws[f"E{i}"] = 1.00; ws[f"E{i}"].fill = inp
    ws[f"F{i}"] = f"=B{i}*E{i}"
    ws[f"G{i}"] = f"=D{i}/F{i}"
    ws[f"H{i}"] = f"=F{i}/C{i}"
    ws[f"I{i}"] = f"=(D{i}*$B$4-F{i})/F{i}"
    ws[f"J{i}"] = f"=D{i}/B{i}"
    ws[f"K{i}"] = f"=D{i}*$B$4/B{i}"
    for col, fmt in zip("DEFGHIJK", ["$#,##0","$0.00","$#,##0","0.00x","$0.00","0%","$0.00","$0.00"]):
        ws[f"{col}{i}"].number_format = fmt

ws["A12"] = "Sensitivity: Paid Search (total) at different CPCs"; ws["A12"].font = bold
for c, h in enumerate(["Assumed CPC $","Spend $","ROAS","CAC $","ROMI","Verdict"], 1):
    ws.cell(row=13, column=c, value=h).font = bold
for j, cpc in enumerate([0.25,0.50,0.75,1.00,1.25,1.50,1.75,2.00,2.50], start=14):
    ws[f"A{j}"] = cpc; ws[f"A{j}"].number_format = "$0.00"
    ws[f"B{j}"] = f"=$B$7*A{j}";            ws[f"B{j}"].number_format = "$#,##0"
    ws[f"C{j}"] = f"=$D$7/B{j}";            ws[f"C{j}"].number_format = "0.00x"
    ws[f"D{j}"] = f"=B{j}/$C$7";            ws[f"D{j}"].number_format = "$0.00"
    ws[f"E{j}"] = f"=($D$7*$B$4-B{j})/B{j}"; ws[f"E{j}"].number_format = "0%"
    ws[f"F{j}"] = f'=IF(E{j}>0,"Profitable","Loss-making")'

for col in range(1, 12):
    ws.column_dimensions[get_column_letter(col)].width = 22

path = "module_2_google_analytics/outputs/roas_scenario.xlsx"
wb.save(path); print("saved:", path)

saved: module_2_google_analytics/outputs/roas_scenario.xlsx


In [3]:
for seg, s, b, r in rows:
    print(f"{seg:28} break-even CPC (revenue): ${r/s:.2f} | (profit @50% margin): ${r*0.5/s:.2f}")

Paid Search (total)          break-even CPC (revenue): $1.72 | (profit @50% margin): $0.86
AW - Dynamic Search Ads      break-even CPC (revenue): $1.74 | (profit @50% margin): $0.87
AW - Accessories             break-even CPC (revenue): $2.63 | (profit @50% margin): $1.31
